# KAN-to-MLP Parameter Matching

In [4]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(42)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Generate KAN and MLP Architecture Candidates



In [8]:
# -------------------------
# Setup & Configurations
# -------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

input_shape = (1, 2)

kan_depths = [1, 2, 3]
kan_widths = [15, 25, 35]
kan_grids = [3, 5, 7]
spline_orders = [2, 3, 4]

# MLP widths to test
mlp_sweep_widths = range(5, 150, 1)


# -------------------------
# Storage
# -------------------------
kan_labels = []
kan_depths_list = []
kan_widths_list = []
kan_grids_list = []
kan_spline_orders_list = []

kan_params_list = []
mlp_matched_params_list = []

mlp_matched_widths = []


for depth in kan_depths:
    for units in kan_widths:
        for grid in kan_grids:
            for spline_order in spline_orders:
                # -------------------------------------------------
                # Build KAN
                # -------------------------------------------------
                model_kan, _ = build_models_KAN(
                    device,
                    hidden_layers=depth,
                    hidden_units=units,
                    grid_size=grid,
                    spline_order=spline_order,
                )

                # -------------------------------------------------
                # Calculate KAN parameters
                # -------------------------------------------------
                _, _, kan_params = calculate_flops(
                    model_kan,
                    input_shape=input_shape,
                    print_results=False,
                    print_detailed=False,
                    output_as_string=False,
                )

                # -------------------------------------------------
                # Find MLP with closest number of parameters
                # -------------------------------------------------
                best_width = None
                best_mlp_params = None
                min_difference = float("inf")

                for m_width in mlp_sweep_widths:
                    model_mlp, _ = build_models(
                        device,
                        hidden_layers=depth,
                        hidden_units=m_width,
                    )

                    _, _, mlp_params = calculate_flops(
                        model_mlp,
                        input_shape=input_shape,
                        print_results=False,
                        print_detailed=False,
                        output_as_string=False,
                    )

                    difference = abs(mlp_params - kan_params)
                    if difference < min_difference:
                        min_difference = difference
                        best_width = m_width
                        best_mlp_params = mlp_params

                # -------------------------------------------------
                # Store results
                # -------------------------------------------------
                kan_labels.append(f"L={depth}, N={units}, G={grid}, k={spline_order}")
                kan_depths_list.append(depth)
                kan_widths_list.append(units)
                kan_grids_list.append(grid)
                kan_spline_orders_list.append(spline_order)
                kan_params_list.append(kan_params)
                mlp_matched_widths.append(best_width)
                mlp_matched_params_list.append(best_mlp_params)

## Determine the KAN Parameter Range


In [10]:
import pandas as pd

# KAN parameter range over all tested depths, widths, grids, and spline orders.
kan_results = pd.DataFrame({
    "hidden_layers": kan_depths_list,
    "hidden_units": kan_widths_list,
    "grid_size": kan_grids_list,
    "spline_order": kan_spline_orders_list,
    "parameters": kan_params_list,
})

kan_min_row = kan_results.loc[kan_results["parameters"].idxmin()]
kan_max_row = kan_results.loc[kan_results["parameters"].idxmax()]
kan_min_params = int(kan_min_row["parameters"])
kan_max_params = int(kan_max_row["parameters"])

print("KAN PARAMETER RANGE")
print("=" * 70)
print(
    "Minimum: "
    f"{kan_min_params:,} parameters "
    f"(L={int(kan_min_row['hidden_layers'])}, "
    f"N={int(kan_min_row['hidden_units'])}, "
    f"G={int(kan_min_row['grid_size'])}, "
    f"k={int(kan_min_row['spline_order'])})"
)
print(
    "Maximum: "
    f"{kan_max_params:,} parameters "
    f"(L={int(kan_max_row['hidden_layers'])}, "
    f"N={int(kan_max_row['hidden_units'])}, "
    f"G={int(kan_max_row['grid_size'])}, "
    f"k={int(kan_max_row['spline_order'])})"
)
print()

# Find MLP widths whose parameter count lies inside the full KAN range.
mlp_range_rows = []
for depth in kan_depths:
    mlp_rows = []
    for width in mlp_sweep_widths:
        model_mlp, _ = build_models(
            device,
            hidden_layers=depth,
            hidden_units=width,
        )
        _, _, mlp_params = calculate_flops(
            model_mlp,
            input_shape=input_shape,
            print_results=False,
            print_detailed=False,
            output_as_string=False,
        )
        mlp_rows.append((width, int(mlp_params)))

    mlp_in_range = [
        (width, params)
        for width, params in mlp_rows
        if kan_min_params <= params <= kan_max_params
    ]

    if mlp_in_range:
        first_width, first_params = mlp_in_range[0]
        last_width, last_params = mlp_in_range[-1]
        width_range = f"{first_width}-{last_width}"
        parameter_range = f"{first_params:,}-{last_params:,}"
    else:
        width_range = "No width in tested range"
        parameter_range = "-"

    mlp_range_rows.append({
        "MLP hidden layers": depth,
        "MLP widths in KAN range": width_range,
        "MLP parameter range": parameter_range,
    })

mlp_range_df = pd.DataFrame(mlp_range_rows)
print("MLP WIDTHS WITHIN THE KAN PARAMETER RANGE")
print("=" * 70)
print(mlp_range_df.to_string(index=False))



KAN PARAMETER RANGE
Minimum: 315 parameters (L=1, N=15, G=3, k=2)
Maximum: 33,215 parameters (L=3, N=35, G=7, k=4)

MLP WIDTHS WITHIN THE KAN PARAMETER RANGE
 MLP hidden layers MLP widths in KAN range MLP parameter range
                 1                  16-149          337-22,947
                 2                  12-127          361-33,021
                 3                  10-104          371-33,177


## Three MLP Configurations

In [15]:
# Recommend one MLP depth for each KAN budget: 1, 2, and 3 layers.
mlp_candidates = []
for depth in kan_depths:
    for width in mlp_sweep_widths:
        model_mlp, _ = build_models(
            device,
            hidden_layers=depth,
            hidden_units=width,
        )
        _, _, mlp_params = calculate_flops(
            model_mlp,
            input_shape=input_shape,
            print_results=False,
            print_detailed=False,
            output_as_string=False,
        )
        mlp_candidates.append({
            "hidden_layers": depth,
            "hidden_units": width,
            "parameters": int(mlp_params),
        })

mlp_candidates_df = pd.DataFrame(mlp_candidates)
intermediate_params = (kan_min_params + kan_max_params) / 2
targets = [
    ("Small (S)", kan_min_params, 1),
    ("Medium (M)", intermediate_params, 2),
    ("Large (L)", kan_max_params, 3),
]

recommendations = []
for label, target, depth in targets:
    depth_candidates = mlp_candidates_df[
        mlp_candidates_df["hidden_layers"] == depth
    ]
    distances = (depth_candidates["parameters"] - target).abs()
    match = depth_candidates.loc[distances.idxmin()]
    recommendations.append({
        "Configuration": label,
        "Target parameters": round(target),
        "MLP hidden layers": depth,
        "MLP hidden units": int(match["hidden_units"]),
        "MLP parameters": int(match["parameters"]),
        "Difference": int(abs(match["parameters"] - target)),
    })

recommendations_df = pd.DataFrame(recommendations)
print("RECOMMENDED MLP CONFIGURATIONS")
print("=" * 80)
print(recommendations_df.to_string(index=False))
print()
print(
    "The small, medium, and large configurations are matched to "
    "1-, 2-, and 3-hidden-layer MLPs, respectively."
)

RECOMMENDED MLP CONFIGURATIONS
Configuration  Target parameters  MLP hidden layers  MLP hidden units  MLP parameters  Difference
    Small (S)                315                  1                15             301          14
   Medium (M)              16765                  2                90           16741          24
    Large (L)              33215                  3               104           33177          38

The small, medium, and large configurations are matched to 1-, 2-, and 3-hidden-layer MLPs, respectively.


## Summary Table


In [19]:
FIGURES_DIR = os.path.join(current_dir, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)


def format_range(values):
    values = sorted(set(values))
    return f"{values[0]}-{values[-1]}" if len(values) > 1 else f"{values[0]}"


# Table A: hyperparameter search space tested for KAN and MLP.
search_space_table = pd.DataFrame(
    [
        {"Hyperparameter": "Hidden layers", "KAN": format_range(kan_depths), "MLP": format_range(kan_depths)},
        {"Hyperparameter": "Hidden units", "KAN": format_range(kan_widths), "MLP": format_range(mlp_sweep_widths)},
        {"Hyperparameter": "Grid size", "KAN": format_range(kan_grids), "MLP": "-"},
        {"Hyperparameter": "Spline order", "KAN": format_range(spline_orders), "MLP": "-"},
    ]
)

# KAN configuration whose parameter count is closest to the min/max midpoint.
kan_medium_row = kan_results.loc[
    (kan_results["parameters"] - intermediate_params).abs().idxmin()
]
kan_medium_params = int(kan_medium_row["parameters"])

# Table B: recommended MLP architectures matched to each KAN parameter configuration.
architecture_matching_table = pd.DataFrame(
    [
        {
            "Configuration": "Small (S)",
            "KAN hidden layers": int(kan_min_row["hidden_layers"]),
            "KAN hidden units": int(kan_min_row["hidden_units"]),
            "KAN grid size": int(kan_min_row["grid_size"]),
            "KAN spline order": int(kan_min_row["spline_order"]),
            "KAN parameters": kan_min_params,
        },
        {
            "Configuration": "Medium (M)",
            "KAN hidden layers": int(kan_medium_row["hidden_layers"]),
            "KAN hidden units": int(kan_medium_row["hidden_units"]),
            "KAN grid size": int(kan_medium_row["grid_size"]),
            "KAN spline order": int(kan_medium_row["spline_order"]),
            "KAN parameters": kan_medium_params,
        },
        {
            "Configuration": "Large (L)",
            "KAN hidden layers": int(kan_max_row["hidden_layers"]),
            "KAN hidden units": int(kan_max_row["hidden_units"]),
            "KAN grid size": int(kan_max_row["grid_size"]),
            "KAN spline order": int(kan_max_row["spline_order"]),
            "KAN parameters": kan_max_params,
        },
    ]
).merge(
    recommendations_df[
        ["Configuration", "MLP hidden layers", "MLP hidden units", "MLP parameters", "Difference"]
    ],
    on="Configuration",
)
architecture_matching_table["Rel. difference (%)"] = (
    100 * architecture_matching_table["Difference"] / architecture_matching_table["KAN parameters"]
).round(2)
architecture_matching_table = architecture_matching_table.drop(columns="Difference")
architecture_matching_table = architecture_matching_table.iloc[:, 1:-1]


print("TABLE: RECOMMENDED MLP ARCHITECTURES MATCHED TO KAN CONFIGURATIONS")
print("=" * 70)
print(architecture_matching_table.to_string(index=False))

search_space_table.to_csv(os.path.join(FIGURES_DIR, "search_space_table.csv"), index=False)
architecture_matching_table.to_csv(os.path.join(FIGURES_DIR, "architecture_matching_table.csv"), index=False)
search_space_table.to_latex(os.path.join(FIGURES_DIR, "search_space_table.tex"), index=False)
architecture_matching_table.to_latex(os.path.join(FIGURES_DIR, "architecture_matching_table.tex"), index=False)

TABLE: RECOMMENDED MLP ARCHITECTURES MATCHED TO KAN CONFIGURATIONS
 KAN hidden layers  KAN hidden units  KAN grid size  KAN spline order  KAN parameters  MLP hidden layers  MLP hidden units  MLP parameters
                 1                15              3                 2             315                  1                15             301
                 3                25              7                 4           17225                  2                90           16741
                 3                35              7                 4           33215                  3               104           33177
